# Sistema de Recomendação utilizando vetores e espaço vetorial

In [1]:
import ast
import nltk
import sklearn
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
pd.options.mode.chained_assignment = None

## Carregando os dados

In [2]:
filmes_df = pd.read_csv("dados/dataset_filmes.csv")

In [3]:
elenco_df = pd.read_csv("dados/dataset_elenco.csv")

In [4]:
filmes_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [5]:
filmes_df.shape

(4803, 20)

In [6]:
elenco_df.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [7]:
elenco_df.shape

(4803, 4)

## Organização dos dados de texto

In [8]:
filmes_df_completo = filmes_df.merge(elenco_df, on='title')

In [9]:
filmes_df_completo.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [10]:
filmes_df_completo.shape

(4809, 23)

In [11]:
filmes_df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [12]:
filmes_df_completo = filmes_df_completo[['movie_id', 'title', 'overview', 'genres', 'keywords', 'vote_average']]

In [13]:
filmes_df_completo.head(10)

,movie_id,title,overview,genres,keywords,vote_average
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",7.2
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",6.9
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",6.3
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",7.6
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",6.1
5,559,Spider-Man 3,The seemingly invincible Spider-Man goes up ag...,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""na...","[{""id"": 851, ""name"": ""dual identity""}, {""id"": ...",5.9
6,38757,Tangled,When the kingdom's most wanted-and most charmi...,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...","[{""id"": 1562, ""name"": ""hostage""}, {""id"": 2343,...",7.4
7,99861,Avengers: Age of Ultron,When Tony Stark tries to jumpstart a dormant p...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 8828, ""name"": ""marvel comic""}, {""id"": ...",7.3
8,767,Harry Potter and the Half-Blood Prince,"As Harry begins his sixth year at Hogwarts, he...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 616, ""name"": ""witch""}, {""id"": 2343, ""n...",7.4
9,209112,Batman v Superman: Dawn of Justice,Fearing the actions of a god-like Super Hero l...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 7002...",5.7


## Análise Exploratória e Limpeza de Dados

In [14]:
filmes_df_completo.shape

(4809, 6)

In [15]:
filmes_df_completo.isnull().sum()

movie_id        0
title           0
overview        3
genres          0
keywords        0
vote_average    0
dtype: int64

In [16]:
filmes_df_completo.dropna(inplace=True)

In [17]:
filmes_df_completo.isnull().sum()

movie_id        0
title           0
overview        0
genres          0
keywords        0
vote_average    0
dtype: int64

In [18]:
filmes_df_completo.duplicated().sum()

np.int64(0)

In [19]:
filmes_df_completo.shape

(4806, 6)

## Processamento de Texto com Abstract Syntax Trees

In [20]:
def string_para_obj(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [21]:
filmes_df_completo['genres'] = filmes_df_completo['genres'].apply(string_para_obj)

In [22]:
filmes_df_completo.head()

,movie_id,title,overview,genres,keywords,vote_average
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",7.2
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",6.9
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",6.3
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",7.6
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",6.1


In [23]:
filmes_df_completo['keywords'] = filmes_df_completo['keywords'].apply(string_para_obj)

In [24]:
filmes_df_completo.head()

,movie_id,title,overview,genres,keywords,vote_average
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",7.2
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",6.9
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",6.3
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",7.6
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",6.1


## Limpeza dos Dados de Texto

In [25]:
filmes_df_completo['overview'] = filmes_df_completo['overview'].apply(lambda x:x.split()) 

In [26]:
filmes_df_completo.head()

,movie_id,title,overview,genres,keywords,vote_average
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",7.2
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",6.9
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",6.3
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",7.6
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",6.1


In [27]:
filmes_df_completo['overview'] = filmes_df_completo['overview'].apply(lambda x:[i.replace(" ", "") for i in x])

In [28]:
filmes_df_completo['genres'] = filmes_df_completo['genres'].apply(lambda x:[i.replace(" ", "") for i in x])

In [29]:
filmes_df_completo['keywords'] = filmes_df_completo['keywords'].apply(lambda x:[i.replace(" ", "") for i in x])

In [30]:
filmes_df_completo.head()

,movie_id,title,overview,genres,keywords,vote_average
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...",7.2
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...",6.9
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...",6.3
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...",7.6
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...",6.1


## Preparando os dados para vetorização em outro espaço vetorial

In [31]:
filmes_df_completo['tags'] = filmes_df_completo['overview'] + \
                             filmes_df_completo['genres'] + \
                             filmes_df_completo['keywords']

In [32]:
filmes_df_completo.head()

,movie_id,title,overview,genres,keywords,vote_average,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...",7.2,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...",6.9,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...",6.3,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...",7.6,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...",6.1,"[John, Carter, is, a, war-weary,, former, mili..."


In [33]:
filmes_df_completo_novo = filmes_df_completo[['movie_id', 'title', 'vote_average', 'tags']]

In [34]:
filmes_df_completo_novo.head()

,movie_id,title,vote_average,tags
0,19995,Avatar,7.2,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,6.9,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,6.3,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,7.6,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,6.1,"[John, Carter, is, a, war-weary,, former, mili..."


In [35]:
filmes_df_completo_novo['tags'] = filmes_df_completo_novo['tags'].apply(lambda x: " ".join(x)) 

In [36]:
filmes_df_completo_novo['tags'] = filmes_df_completo_novo['tags'].apply(lambda x:x.lower()) 

In [37]:
filmes_df_completo_novo.head(10)

,movie_id,title,vote_average,tags
0,19995,Avatar,7.2,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,6.9,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,6.3,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,7.6,following the death of district attorney harve...
4,49529,John Carter,6.1,"john carter is a war-weary, former military ca..."
5,559,Spider-Man 3,5.9,the seemingly invincible spider-man goes up ag...
6,38757,Tangled,7.4,when the kingdom's most wanted-and most charmi...
7,99861,Avengers: Age of Ultron,7.3,when tony stark tries to jumpstart a dormant p...
8,767,Harry Potter and the Half-Blood Prince,7.4,"as harry begins his sixth year at hogwarts, he..."
9,209112,Batman v Superman: Dawn of Justice,5.7,fearing the actions of a god-like super hero l...


## Parse e Vetorização

In [38]:
parser_ps = PorterStemmer()

In [39]:
def apply_stem(text):
    y = []
    for i in text.split():
        y.append(parser_ps.stem(i))

    return " ".join(y)

In [40]:
filmes_df_completo_novo['tags'] = filmes_df_completo_novo['tags'].apply(apply_stem)

In [41]:
filmes_df_completo_novo.head(10)

,movie_id,title,vote_average,tags
0,19995,Avatar,7.2,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,6.9,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,6.3,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,7.6,follow the death of district attorney harvey d...
4,49529,John Carter,6.1,"john carter is a war-weary, former militari ca..."
5,559,Spider-Man 3,5.9,the seemingli invinc spider-man goe up against...
6,38757,Tangled,7.4,when the kingdom' most wanted-and most charmin...
7,99861,Avengers: Age of Ultron,7.3,when toni stark tri to jumpstart a dormant pea...
8,767,Harry Potter and the Half-Blood Prince,7.4,"as harri begin hi sixth year at hogwarts, he d..."
9,209112,Batman v Superman: Dawn of Justice,5.7,fear the action of a god-lik super hero left u...


In [42]:
cv = CountVectorizer(max_features = 5000, stop_words = 'english')

In [43]:
vectors = cv.fit_transform(filmes_df_completo_novo['tags']).toarray()

In [44]:
len(cv.get_feature_names_out())

5000

In [45]:
type(vectors)

numpy.ndarray

In [46]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [47]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [48]:
vectors[1]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

## Cálculo de Distância Entre os Vetores e Normalização da Nota

In [49]:
similaridades = cosine_similarity(vectors)

In [50]:
scaler = MinMaxScaler()
filmes_df_completo_novo['vote_average_norm'] = scaler.fit_transform(filmes_df_completo_novo[['vote_average']])

In [51]:
filmes_df_completo_novo.head()

,movie_id,title,vote_average,tags,vote_average_norm
0,19995,Avatar,7.2,"in the 22nd century, a parapleg marin is dispa...",0.72
1,285,Pirates of the Caribbean: At World's End,6.9,"captain barbossa, long believ to be dead, ha c...",0.69
2,206647,Spectre,6.3,a cryptic messag from bond’ past send him on a...,0.63
3,49026,The Dark Knight Rises,7.6,follow the death of district attorney harvey d...,0.76
4,49529,John Carter,6.1,"john carter is a war-weary, former militari ca...",0.61


## Construindo o Sistema de Recomendação

In [67]:
def sistema_recomendacao(movie, peso_similaridade = 0.85, peso_nota = 0.15):
    # Obtém o índice do filme passado como argumento
    index = filmes_df_completo_novo[filmes_df_completo_novo['title'] == movie].index[0]

    # Pega as distâncias/similaridades apenas para este filme
    sim_scores = similaridades[index]

    # Lista para armazenar o score final de cada filme
    pontuacoes_finais = []
    
    for i, sim_score in enumerate(sim_scores):
        # Ignora o filme que foi passado como referência 
        if i != index:
            # Pega a nota normalizada desse filme
            nota_norm = filmes_df_completo_novo.iloc[i]['vote_average_norm']

            # Calcula o score final (média ponderada)
            score_final = (sim_score * peso_similaridade) + (nota_norm * peso_nota)

            # Salva o índice original e o score final
            pontuacoes_finais.append((i, score_final))

    # Ordena a lista com base no score final (do maior para o menor)
    pontuacoes_finais = sorted(pontuacoes_finais, reverse=True, key=lambda x: x[1])

    # Imprime os 5 melhores resultados
    print(f"Recomendações baseadas em {movie}:\n")
    for i in pontuacoes_finais[0:5]:
        filme_recomendado = filmes_df_completo_novo.iloc[i[0]].title
        score_obtido = i[1]
        print(f"- {filme_recomendado} (Score Híbrido: {score_obtido:.4f})")

In [68]:
sistema_recomendacao('Jurassic World')

Recomendações baseadas em Jurassic World:

- Jurassic Park (Score Híbrido: 0.5415)
- The Lost World: Jurassic Park (Score Híbrido: 0.4597)
- Walking With Dinosaurs (Score Híbrido: 0.3279)
- Guardians of the Galaxy (Score Híbrido: 0.2981)
- Jurassic Park III (Score Híbrido: 0.2975)


In [69]:
sistema_recomendacao('Avengers: Age of Ultron')

Recomendações baseadas em Avengers: Age of Ultron:

- Iron Man (Score Híbrido: 0.4030)
- Iron Man 2 (Score Híbrido: 0.3953)
- Iron Man 3 (Score Híbrido: 0.3940)
- The Wolverine (Score Híbrido: 0.3806)
- Ant-Man (Score Híbrido: 0.3790)


In [70]:
sistema_recomendacao('The Hobbit: The Battle of the Five Armies')

Recomendações baseadas em The Hobbit: The Battle of the Five Armies:

- The Hobbit: An Unexpected Journey (Score Híbrido: 0.4976)
- The Hobbit: The Desolation of Smaug (Score Híbrido: 0.4810)
- The Lord of the Rings: The Fellowship of the Ring (Score Híbrido: 0.4257)
- The Lord of the Rings: The Two Towers (Score Híbrido: 0.4177)
- The Lord of the Rings: The Return of the King (Score Híbrido: 0.3952)


In [71]:
sistema_recomendacao("Pirates of the Caribbean: At World's End")

Recomendações baseadas em Pirates of the Caribbean: At World's End:

- Pirates of the Caribbean: Dead Man's Chest (Score Híbrido: 0.3789)
- Pirates of the Caribbean: On Stranger Tides (Score Híbrido: 0.3077)
- Pirates of the Caribbean: The Curse of the Black Pearl (Score Híbrido: 0.2991)
- Life of Pi (Score Híbrido: 0.2906)
- 20,000 Leagues Under the Sea (Score Híbrido: 0.2882)
